# 🎮 🛠️ Practical: Using a 6-billion-parameter LLM from Hugging Face

In this practical we download a real 6B-parameter language model, run it on a GPU, and make it solve a task: **classifying movie reviews as positive or negative**. We never train anything. We only change the *prompt*, and watch how the model's behaviour changes.

**Before you start:** in Colab go to `Runtime > Change runtime type` and choose a **T4 GPU**.

By the end you will have seen:
1. How to load a model and tokenizer from Hugging Face
2. How a model turns text into tokens and tokens back into text
3. Zero-shot prompting (just ask)
4. Few-shot prompting (show examples, then ask)

## Step 1: Install and import

In [ ]:
# transformers = Hugging Face's library for loading models
# accelerate   = lets the model be placed on the GPU automatically
# bitsandbytes = lets us shrink the model to 4-bit numbers so it fits in a free GPU
!pip install -q transformers accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Check that we actually have a GPU. If this prints False, fix the runtime type first.
print("GPU available:", torch.cuda.is_available())

## Step 2: Load the model

We use **Yi-6B**, an open model with about 6 billion parameters.

A 6B model stored as 16-bit numbers needs about 12 GB of memory. The free Colab GPU has about 15 GB, which is tight. So we store the weights as **4-bit** numbers instead (about 4 GB). This costs a little accuracy but makes it fit comfortably.

The first run downloads roughly 12 GB, so expect a few minutes.

In [ ]:
model_name = "01-ai/Yi-6B"

# The tokenizer converts text <-> numbers. Every model has its own, so we must load the matching one.
tokenizer = AutoTokenizer.from_pretrained(model_name)

# This config tells Hugging Face to squeeze the weights down to 4 bits when loading.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,  # do the maths in 16-bit for stability
)

# device_map="auto" puts the model on the GPU for us.
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
)

# Count the parameters, so we can see that the model really is ~6 billion.
total_parameters = sum(p.numel() for p in model.parameters())
print("Parameters (billions):", total_parameters / 1e9)

config.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B / 1.03MB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/3.60M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

## Step 3: What does the tokenizer do?

A language model never sees letters or words. It sees **token IDs**, which are integers. Let's look at one sentence.

In [ ]:
sentence = "The film was a delight."

# Turn the text into a list of integer IDs.
token_ids = tokenizer.encode(sentence)
print("Token IDs:", token_ids)

# Look at what piece of text each ID stands for.
for token_id in token_ids:
    print(token_id, "->", repr(tokenizer.decode([token_id])))

**Question:** is every word one token? What happens to the full stop?

## Step 4: Make the model generate text

This model is a **base model**: it was trained only to predict the next token. It does not "follow instructions". If we give it the start of some text, it continues it. Let's see.

In [ ]:
def generate(prompt, max_new_tokens=20):
    # Turn the prompt into token IDs and move them to the GPU (as a PyTorch tensor).
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Ask the model to continue the text.
    # do_sample=False means "always pick the most likely next token", so results are repeatable.
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )

    # The output contains the prompt AND the new tokens. We slice off the prompt part.
    number_of_prompt_tokens = inputs["input_ids"].shape[1]
    new_ids = output_ids[0][number_of_prompt_tokens:]

    # Turn the new IDs back into text.
    return tokenizer.decode(new_ids, skip_special_tokens=True)

print(generate("The capital of France is"))

## Step 5: Our task and a tiny test set

Eight short reviews with the correct label. Small enough to read in full and check by hand.

In [ ]:
test_reviews = [
    ("An absolute masterpiece, I loved every minute.", "positive"),
    ("Dull, predictable and far too long.", "negative"),
    ("The acting was superb and the story moved me.", "positive"),
    ("I want my two hours back.", "negative"),
    ("A charming film that made the whole cinema smile.", "positive"),
    ("The plot made no sense and the jokes fell flat.", "negative"),
    ("Not bad at all, I would happily watch it again.", "positive"),
    ("It was not good. The ending was a mess.", "negative"),
]
print("Number of reviews:", len(test_reviews))

## Step 6: Zero-shot prompting

**Zero-shot** means we describe the task and give no examples. Since this is a base model, we write the prompt so that the natural continuation is the answer.

In [ ]:
def make_zero_shot_prompt(review):
    # We end the prompt with "Sentiment:" so the most natural next word is the label.
    return (
        "Classify the sentiment of the movie review as positive or negative.\n\n"
        "Review: " + review + "\n"
        "Sentiment:"
    )

# Look at one prompt exactly as the model sees it.
print(make_zero_shot_prompt(test_reviews[0][0]))

In [ ]:
def evaluate(make_prompt):
    number_correct = 0

    for review, true_label in test_reviews:
        prompt = make_prompt(review)

        # We only need one or two words, so generate very few tokens.
        answer = generate(prompt, max_new_tokens=3).strip().lower()

        # Our simple rule: the prediction is whichever label the answer starts with.
        if answer.startswith("positive"):
            predicted_label = "positive"
        elif answer.startswith("negative"):
            predicted_label = "negative"
        else:
            predicted_label = "unclear"

        # Print every case so we can see exactly where the model goes wrong.
        print(true_label, "|", predicted_label, "|", review)

        if predicted_label == true_label:
            number_correct += 1

    print("\nAccuracy:", number_correct, "/", len(test_reviews))

evaluate(make_zero_shot_prompt)

## Step 7: Few-shot prompting

**Few-shot** means we put a few solved examples in the prompt before the real question. The model is not being trained; it just sees a pattern and continues it.

In [ ]:
examples = [
    ("A wonderful, heartwarming story.", "positive"),
    ("Boring and badly acted.", "negative"),
    ("I laughed from start to finish.", "positive"),
    ("A complete waste of money.", "negative"),
]

def make_few_shot_prompt(review):
    prompt = "Classify the sentiment of the movie review as positive or negative.\n\n"

    # Add each solved example in exactly the same format as the final question.
    for example_review, example_label in examples:
        prompt = prompt + "Review: " + example_review + "\n"
        prompt = prompt + "Sentiment: " + example_label + "\n\n"

    # Finally add the real review, and leave the label blank for the model to fill in.
    prompt = prompt + "Review: " + review + "\n"
    prompt = prompt + "Sentiment:"
    return prompt

print(make_few_shot_prompt(test_reviews[0][0]))

In [ ]:
evaluate(make_few_shot_prompt)

**Question:** did accuracy change? Which reviews changed from wrong to right, or the reverse?

## Exercises

**Exercise 1 (prompt wording).** Change the instruction line in `make_few_shot_prompt` (for example to `"Is this review good or bad?"`) and change the labels in `examples` to `good` / `bad`. Update `evaluate` to match. Does accuracy move?

**Exercise 2 (order matters).** Reverse the order of `examples`, so the negatives come first. Re-run. Large models are known to be sensitive to this. Do you see it?

**Exercise 3 (your own task).** Replace the sentiment task with a different one that you design, using the same few-shot pattern. Suggestions:
- Label a sentence as `question` or `statement`
- Translate a word from English to French (examples: `cat -> chat`)
- Extract the city from a sentence (`I flew to Paris on Monday.` -> `Paris`)

**Exercise 4 (limits).** Write three reviews that you think will fool the model (sarcasm, double negatives, mixed opinions). Add them to `test_reviews` and check.

## Check your understanding

1. We never updated the model's weights. So where did the "learning" in few-shot prompting happen?
2. Why did we use `do_sample=False`? What would change if we set it to `True`?
3. The model is 6B parameters stored in 4 bits each. Roughly how many GB is that? (Work it out by hand, then compare with 12 GB for 16-bit.)
4. A base model continues text. What would you need to change to use an *instruction-tuned* model instead?

# 🛠️ Activity

## Check your understanding: where does the "learning" in few-shot prompting happen?

We never updated the model's weights, so the learning happens somewhere else: in the model's **activations**, at inference time. This is called **in-context learning**.

**How it works.** The weights were fixed during pretraining. When we put solved examples in the prompt, the model reads them together with our review. Attention lets every later token look back at them. The examples change the model's internal activations, and so the probability of each next token. After `Sentiment:`, the continuation `positive` or `negative` becomes very likely, because that is the pattern the context sets up.

**Nothing is stored.** If we call the model again without the examples, it has "forgotten" the task. The learning lasts only as long as the prompt does.

**What is actually being learned?** The ability to continue patterns and to recognise sentiment was already in the weights from pretraining. The examples mostly tell the model *which* task we want and *what format* the answer should take. Researchers still disagree on how to describe this:
- One view: the model is **recognising** a task it already knows.
- Another view: the transformer carries out something like a small learning algorithm inside its forward pass.

**Look at your own results.** If zero-shot and few-shot both scored 8/8, the task was easy enough that the examples had no room to help. For an easy task, few-shot adds little. It matters more for harder tasks or unusual output formats. Try Exercises 3 and 4 to see a difference.

### A note on the parameter count

You may notice that the model prints about **3.3 billion** parameters, not 6 billion. This is a side effect of 4-bit loading: the `bitsandbytes` library packs two 4-bit weights into each stored element, so `numel()` undercounts by roughly half. The model still has about 6 billion parameters.